# 01 Data Extraction And Audit

This notebook opens the thesis package the way a reviewer should: first by understanding where the dataset came from, what was packaged locally, and how the final modeling subsets were defined.

**Questions answered here**
- What data are inside the thesis package?
- How were hybrid, paired-labeled, and paired-unlabeled subsets defined?
- Why is `recording_key` the leakage boundary?
- What did the extraction and audit phase actually establish before modeling began?


In [1]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

def _find_thesis_root():
    cwd = Path.cwd().resolve()
    direct = [cwd, *cwd.parents]
    nested = [candidate / "thesis" for candidate in direct]
    for candidate in [*direct, *nested]:
        if (candidate / "THESIS_BLUEPRINT.md").exists() and (candidate / "src" / "qc_thesis" / "__init__.py").exists():
            return candidate
    raise FileNotFoundError("Could not find thesis root from notebook session")

ROOT = _find_thesis_root()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qc_thesis import *

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
apply_thesis_style()

fig_dir, table_dir = notebook_output_dirs("01_data_extraction_and_audit")
dataset_summary = load_dataset_summary()
computed_summary = compute_dataset_summary()
partition_table = build_dataset_partition_table()
target_missingness = build_target_missingness_table()
family_coverage = build_family_coverage_table()
feature_inventory = build_feature_inventory_table()
manifest = json.loads((ROOT / "data/frozen_inputs/manifest.json").read_text())


## 1. Packaged provenance

The thesis package contains both the exploratory raw parquet and frozen benchmark inputs. The point is not only to show the final tables, but to show the entire route from extraction to evaluation.


In [2]:
display(pd.DataFrame(dataset_summary.items(), columns=["packaged_item", "value"]))
display(pd.DataFrame(computed_summary.items(), columns=["recomputed_item", "value"]))
display(pd.DataFrame(manifest.items(), columns=["artifact_group", "path"]))
save_table(pd.DataFrame(dataset_summary.items(), columns=["item", "value"]), table_dir, "packaged_dataset_summary")
save_table(pd.DataFrame(computed_summary.items(), columns=["item", "value"]), table_dir, "recomputed_dataset_summary")


,packaged_item,value
0,parquet_source_path,/Users/paulruiz/Documents/Predicting_Good_Unit...
1,rows,33206
2,unique_recordings,61
3,unique_sorters,9
4,fmiss_missing_rows,14546
5,fmiss_non_missing_rows,18660


,recomputed_item,value
0,total_rows,33206
1,unique_recordings,61
2,unique_sorters,9
3,feature_columns,184
4,hybrid_labeled_rows,24224
5,paired_rows,8982
6,paired_labeled_rows,207
7,paired_unlabeled_rows,8775


,artifact_group,path
0,frozen_inputs_root,/Users/paulruiz/Documents/Predicting_Good_Unit...
1,legacy_final_holdout_results,benchmarks/legacy_final_holdout_results.csv
2,ssl_metrics,benchmarks/ssl_metrics.csv
3,mmd_metrics,benchmarks/mmd_metrics.csv
4,workbench_r2_metrics,benchmarks/workbench_r2_metrics.csv
5,workbench_r2_predictions,benchmarks/workbench_r2_predictions.csv
6,fpos_backend_metrics,benchmarks/fpos_backend_metrics.csv
7,fpos_backend_predictions,benchmarks/fpos_backend_predictions.csv
8,fpos_family_robustness_metrics,benchmarks/fpos_family_robustness_metrics.csv
9,fpos_family_robustness_predictions,benchmarks/fpos_family_robustness_predictions.csv


PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/thesis/tables/01_data_extraction_and_audit/recomputed_dataset_summary.csv')

## 2. Subset definitions

The thesis uses three operational subsets:

- **hybrid labeled**: large source domain with labels
- **paired labeled**: small matched target-domain supervision
- **paired unlabeled**: real target-domain support used only structurally

This table is the foundation for every later benchmark and ablation.


In [3]:
display(partition_table)
display(target_missingness)
save_table(partition_table, table_dir, "dataset_partition_table")
save_table(target_missingness, table_dir, "target_missingness")


,subset,rows,recordings,families
0,full,33206,61,6
1,hybrid_labeled,24224,30,1
2,paired_total,8982,31,5
3,paired_labeled,207,29,5
4,paired_unlabeled,8775,31,5


,target,missing_rows,missing_fraction,non_missing_rows
0,fpos,0,0.000000,33206
1,fmiss,14546,0.438053,18660
2,accuracy,0,0.000000,33206


PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/thesis/tables/01_data_extraction_and_audit/target_missingness.csv')

## 3. Family and feature coverage

Before modeling, the important question was whether the paired domain behaved like one domain or a mixture of regimes. Family coverage and feature inventory already hint that the answer is “mixture”.


In [4]:
display(family_coverage)
display(feature_inventory)
save_table(family_coverage, table_dir, "paired_family_coverage")
save_table(feature_inventory, table_dir, "feature_inventory")


,study_set,rows,recordings,sorters,labeled_rows
0,PAIRED_ENGLISH,3151,15,7,71
1,PAIRED_MEA64C_YGER,2997,4,9,36
2,PAIRED_KAMPFF,1743,4,8,35
3,PAIRED_BOYDEN,907,4,8,43
4,PAIRED_CRCNS_HC1,184,4,8,22


,feature_block,feature_count
0,numeric_total,224
1,waveform_bins,30
2,acg_bins,40
3,context_only,40
4,full_context,264
5,reduced_latent,49
6,waveform_embedding_view,90


PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/thesis/tables/01_data_extraction_and_audit/feature_inventory.csv')

## 4. Why the split protocol is strict

The benchmark is optimized on **recording-disjoint** transfer. A second stress protocol leaves an entire paired family unseen. This notebook makes that separation explicit before any performance claims are shown.


In [5]:
bundle = build_dataset_bundle(with_context=False)
split = recording_disjoint_split(bundle.paired_labeled_df, test_rows_target=100, random_state=42)
lofo_rows = []
for family_name in sorted(bundle.paired_labeled_df["study_set"].dropna().unique()):
    family_split = leave_one_family_out(bundle.paired_labeled_df, family=family_name)
    lofo_rows.append({
        "held_out_family": family_name,
        "train_rows": len(family_split.train_df),
        "test_rows": len(family_split.test_df),
        "train_recordings": family_split.train_df["recording_key"].nunique(),
        "test_recordings": family_split.test_df["recording_key"].nunique(),
    })
lofo_table = pd.DataFrame(lofo_rows).sort_values("held_out_family").reset_index(drop=True)
protocol_summary = pd.DataFrame([
    {
        "protocol": "recording_disjoint_main",
        "train_rows": len(split.train_df),
        "test_rows": len(split.test_df),
        "train_recordings": split.train_df["recording_key"].nunique(),
        "test_recordings": split.test_df["recording_key"].nunique(),
    }
])
display(protocol_summary)
display(lofo_table)
save_table(protocol_summary, table_dir, "recording_disjoint_protocol_summary")
save_table(lofo_table, table_dir, "family_held_out_protocol_summary")


,protocol,train_rows,test_rows,train_recordings,test_recordings
0,recording_disjoint_main,146,61,23,6


,held_out_family,train_rows,test_rows,train_recordings,test_recordings
0,PAIRED_BOYDEN,164,43,25,4
1,PAIRED_CRCNS_HC1,185,22,25,4
2,PAIRED_ENGLISH,136,71,16,13
3,PAIRED_KAMPFF,172,35,25,4
4,PAIRED_MEA64C_YGER,171,36,25,4


PosixPath('/Users/paulruiz/Documents/Predicting_Good_Units/thesis/tables/01_data_extraction_and_audit/family_held_out_protocol_summary.csv')

In [6]:
display(Markdown(
    f"""
## Key takeaways

- The thesis package contains **{partition_table.loc[partition_table['subset'] == 'full', 'rows'].iloc[0]:,} rows** in total.
- Only **{partition_table.loc[partition_table['subset'] == 'paired_labeled', 'rows'].iloc[0]} paired labeled rows** are available for direct supervised adaptation.
- The benchmark therefore depends on transfer and conservative use of **{partition_table.loc[partition_table['subset'] == 'paired_unlabeled', 'rows'].iloc[0]:,} paired unlabeled rows**.
- `fmiss` has the strongest missing-label burden, which already suggests that `fmiss` will be scientifically harder than `fpos`.
"""
))



## Key takeaways

- The thesis package contains **33,206 rows** in total.
- Only **207 paired labeled rows** are available for direct supervised adaptation.
- The benchmark therefore depends on transfer and conservative use of **8,775 paired unlabeled rows**.
- `fmiss` has the strongest missing-label burden, which already suggests that `fmiss` will be scientifically harder than `fpos`.
